# Stage 18D: cross-fitted learned donor ranker

Stage 18Cの固定top-4 donor retrievalをcontrolにし、target-free特徴だけで候補donorを順位付けします。評価foldはtargetとしてもdonorとしても学習から除外します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 固定artifact

Stage 18Cを新しいcontrolとして追加します。Stage 18Dは候補特徴の作成とranked retrievalの2段階で、途中の`donor_training_rows.parquet`は再実行時に利用されます。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage17a_run = artifact_dir / 'stage17_public_replay_full_v002'
stage17b_run = artifact_dir / 'stage17b_selector_replay_full_v001'
stage18c_run = artifact_dir / 'stage18c_all_cut_branch_retrieval_full_v001'
assert (stage16b_run / 'donor_graph.parquet').is_file()
assert (stage17a_run / 'replay_predictions.parquet').is_file()
assert (stage17b_run / 'selector_predictions.parquet').is_file()
assert (stage18c_run / 'summary.json').is_file()
print(stage16b_run, stage17a_run, stage17b_run, stage18c_run, sep='\n')

## cross-fitted donor ranking

全3,865 cuts × 最大12 donorsの特徴とlabelを作るため時間がかかります。CPUで実行できます。suffix TVTはlabelだけに使い、推論特徴には入りません。

In [ ]:
RUN_ID = 'stage18d_learned_donor_ranker_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-donor-ranker',
        '--config', 'configs/experiment/stage18d_learned_donor_ranker.yaml',
        '--stage16b-run', str(stage16b_run), '--stage17a-run', str(stage17a_run),
        '--stage17b-run', str(stage17b_run), '--stage18c-run', str(stage18c_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage18d_complete': summary['stage18d_complete'],
    'promoted_to_full_ranker_training': summary['promoted_to_full_ranker_training'],
    'cuts': summary['cuts'],
    'candidate_rows': summary['candidate_rows'],
    'coverage_fraction': summary['coverage_fraction'],
    'oof_score_spearman': summary['oof_score_spearman'],
    'oracle_top1_recall': summary['oracle_top1_recall'],
    'mean_selected_donor_overlap': summary['mean_selected_donor_overlap'],
    'learned_vs_fixed': summary['learned_vs_fixed'],
    'learned_vs_fixed_branch_folds': summary['learned_vs_fixed_branch_folds'],
    'learned_vs_strong_base': summary['learned_vs_strong_base'],
    'bootstrap_95pct': summary['bootstrap_95pct'],
    'gates': summary['gates'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。Stage 18C固定retrievalより有意に改善した場合だけ、test推論用の全data rankerを作ります。まだKaggle提出は行いません。